# 00 · Anonymisation

Reads `data/données.xlsx`, generates a deterministic anonymous ID per student
(SHA-256 of normalised name + promo), and exports `data/profil_etudiants.csv`.

> ⚠️ Run **once** before everything else. Re-running is safe (deterministic).

In [1]:
import hashlib, unicodedata
import pandas as pd
from pathlib import Path
import sys
sys.path.insert(0, "..")
if 'config' in sys.modules:
    del sys.modules['config']
from config import DATA_DIR

INPUT  = DATA_DIR / "raw" / "profils.xlsx"
OUTPUT = DATA_DIR / "parsed" / "profils_anonymes.csv"



In [2]:
def normalize(s: str) -> str:
    """Lowercase + strip accents."""
    if not s: return ""
    return "".join(
        c for c in unicodedata.normalize("NFD", s.strip().lower())
        if unicodedata.category(c) != "Mn"
    )

def make_id(nom: str, prenom: str, promo) -> str:
    """10-char hex digest — stable across runs."""
    key = f"{normalize(nom)}_{normalize(prenom)}_{int(float(promo))}"
    return hashlib.sha256(key.encode()).hexdigest()[:10]


In [3]:
df = pd.read_excel(INPUT)
print(f"Rows: {len(df)}  |  Columns: {list(df.columns)}")
df.head()


Rows: 143  |  Columns: ['Promotion', 'Nom', 'Prénom', 'Genre', 'origine', 'Licence/ Pisi', 'Bac']


,Promotion,Nom,Prénom,Genre,origine,Licence/ Pisi,Bac
0,1,ABID,OUSSEMA,M,Sfax,Pisi,science
1,1,ALIMI,MOHAMED,M,Sfax,Pisi,math
2,1,ALOULOU,HEDI,M,Sfax,Pisi,math
3,1,AMINE,HENTATI,M,Sfax,Pisi,science
4,1,BARGOUGUI,MONTASSAR,M,Sfax,Pisi,math


In [4]:
df["id_anon"] = df.apply(
    lambda r: make_id(r["Nom"], r["Prénom"], r["Promotion"]), axis=1
)

# Save id↔name mapping BEFORE dropping names
MAPPING = DATA_DIR / "parsed" / "mapping_id_noms.csv"
mapping_df = df[["id_anon", "Nom", "Prénom", "Promotion"]].copy()
mapping_df.to_csv(MAPPING, index=False)
print(f"🗂️  Mapping saved ({len(mapping_df)} rows) → {MAPPING}")

df = df.drop(columns=["Nom", "Prénom"])
df = df[["id_anon"] + [c for c in df.columns if c != "id_anon"]]

# Sanity checks
assert df["id_anon"].nunique() == len(df), "Duplicate IDs detected!"
print(f"Promotions : {sorted(df['Promotion'].unique())}")
print(f"Students   : {len(df)}")
df.head()


🗂️  Mapping saved (143 rows) → /home/mayna/fss/data/parsed/mapping_id_noms.csv
Promotions : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Students   : 143


,id_anon,Promotion,Genre,origine,Licence/ Pisi,Bac
0,a467c6574a,1,M,Sfax,Pisi,science
1,5e305db688,1,M,Sfax,Pisi,math
2,fe746363c4,1,M,Sfax,Pisi,math
3,0b01093cfa,1,M,Sfax,Pisi,science
4,afaf20ddb8,1,M,Sfax,Pisi,math


In [5]:
# Normalise categorical columns before saving
for col in ["Genre", "origine", "Licence/ Pisi", "Bac"]:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

df.to_csv(OUTPUT, index=False)
print(f"✅ Saved {len(df)} students → {OUTPUT}")


✅ Saved 143 students → /home/mayna/fss/data/parsed/profils_anonymes.csv
